In [50]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm.auto import tqdm
import cloudscraper
import time
import sqlite3
from sqlalchemy import create_engine

In [3]:
project_root = Path.cwd().parent

# Define the relative path to the news data
# Path structure: data/raw/news/gdelt_war_data.csv
data_path = project_root / "data" / "raw" / "news" / "gdelt_war_data.csv"

df = pd.read_csv(data_path)
df.head()

,event_date,source,url,themes
0,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_FNCACT_CHANCELLOR,17;TAX_FNCACT_CHANCELLOR..."
1,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_ETHNICITY_AMERICANS,5192;WB_508_POWER_SYST..."
2,2026-03-03,theguardian.com,https://www.theguardian.com/stage/2026/mar/03/...,"CLOSURE,2139;TAX_ETHNICITY_KOREAN,1693;TAX_WOR..."
3,2026-03-03,kyivindependent.com,https://kyivindependent.com/ukraine-backs-us-i...,"TAX_ETHNICITY_IRANIANS,2755;TAX_ETHNICITY_IRAN..."
4,2026-03-03,theguardian.com,https://www.theguardian.com/world/2026/mar/03/...,"TAX_FNCACT_AUTHORITIES,2679;EPU_POLICY_AUTHORI..."


In [4]:
df.loc[0, 'themes']

'TAX_FNCACT_CHANCELLOR,17;TAX_FNCACT_CHANCELLOR,1183;TAX_ETHNICITY_AMERICAN,1076;DRONES,2482;DRONES,2664;DRONES,2808;TAX_ETHNICITY_UKRAINIAN,2956;TAX_WORLDLANGUAGES_UKRAINIAN,2956;WB_2024_ANTI_CORRUPTION_AUTHORITIES,1832;WB_696_PUBLIC_SECTOR_MANAGEMENT,1832;WB_840_JUSTICE,1832;WB_2025_INVESTIGATION,1832;WB_831_GOVERNANCE,1832;WB_832_ANTI_CORRUPTION,1832;WB_1014_CRIMINAL_JUSTICE,1832;MILITARY,1789;FUELPRICES,3469;TAX_FNCACT_SPOKESPERSON,3181;SEIZE,1499;SEIZE,1763;TAX_ECON_PRICE,3469;TAX_FNCACT_LEADERS,3200;MEDIA_MSM,575;MEDIA_MSM,1198;WB_2299_PIPELINES,3026;WB_539_OIL_AND_GAS_POLICY_STRATEGY_AND_INSTITUTIONS,3026;WB_507_ENERGY_AND_EXTRACTIVES,3026;WB_548_PPP_IN_OIL_AND_GAS,3026;ARMEDCONFLICT,388;ARMEDCONFLICT,727;ARMEDCONFLICT,968;ARMEDCONFLICT,2608;EPU_CATS_NATIONAL_SECURITY,388;EPU_CATS_NATIONAL_SECURITY,727;EPU_CATS_NATIONAL_SECURITY,968;EPU_CATS_NATIONAL_SECURITY,2608;MARITIME_INCIDENT,1492;MARITIME_INCIDENT,2119;MARITIME,1492;MARITIME,2119;MANMADE_DISASTER_IMPLIED,1492;MANMADE_DISA

In [5]:
df.loc[0, 'url']

'https://www.theguardian.com/world/2026/mar/04/ukraine-war-briefing-merz-tells-trump-ukraine-must-not-give-up-more-territory'

When I first looked at the GDELT data, I realized the 'themes' column was a mess. It's just a long string of tags like TAX_ETHNICITY_UKRAINIAN. This is good for machines, but it doesn't tell me what actually happened.

I need real headlines to understand the context. So, I decided to filter the data and write a small scraper to get titles directly from news websites.

In [6]:
df.source.unique()

array(['theguardian.com', 'kyivindependent.com', 'bbc.co.uk', 'bbc.com',
       'kelownacapnews.com', 'nytimes.com', 'apnews.com'], dtype=object)

In [7]:
df[df.source == 'kyivindependent.com'].url.iloc[0]

'https://kyivindependent.com/ukraine-backs-us-israeli-war-on-iran-as-zelensky-hopes-to-curry-favor-with-trump/'

In [8]:
def fetch_article_headline(url):
    """
    Scrapes the main headline from a news article URL.
    """
    # Headers are mandatory to avoid 403 Forbidden errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9'
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')

        # Strategy 1: Look for Open Graph metadata (highly reliable for clean titles)
        og_title = soup.find("meta", property="og:title")
        if og_title and og_title.get("content"):
            return og_title["content"].strip()

        # Strategy 2: Look for the primary H1 tag
        h1_tag = soup.find("h1")
        if h1_tag:
            return h1_tag.get_text(strip=True)

        # Strategy 3: Fallback to the standard HTML title tag
        if soup.title:
            return soup.title.get_text(strip=True)

        return "Headline not found"

    except requests.exceptions.RequestException as e:
        return f"Request Error: {e}"
    except Exception as e:
        return f"Unexpected Error: {e}"


In [9]:
df_test = df.head(10).copy()

In [10]:
df_test['headlines'] = df_test.url.apply(fetch_article_headline)

In [11]:
df_test.head()

,event_date,source,url,themes,headlines
0,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_FNCACT_CHANCELLOR,17;TAX_FNCACT_CHANCELLOR...",Ukraine war briefing: Merz tells Trump Ukraine...
1,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_ETHNICITY_AMERICANS,5192;WB_508_POWER_SYST...",Middle East conflict offers economic lifeline ...
2,2026-03-03,theguardian.com,https://www.theguardian.com/stage/2026/mar/03/...,"CLOSURE,2139;TAX_ETHNICITY_KOREAN,1693;TAX_WOR...",Ukraine Unbroken review – five searing dramas ...
3,2026-03-03,kyivindependent.com,https://kyivindependent.com/ukraine-backs-us-i...,"TAX_ETHNICITY_IRANIANS,2755;TAX_ETHNICITY_IRAN...",How will Trump's war against Iran impact Ukraine?
4,2026-03-03,theguardian.com,https://www.theguardian.com/world/2026/mar/03/...,"TAX_FNCACT_AUTHORITIES,2679;EPU_POLICY_AUTHORI...",Ukraine war briefing: Russia’s army records sl...


In [12]:
df_test.headlines.iloc[5]

'What to do if you’re under attack by Iranian Shahed drones'

To ensure high data integrity and minimize "geopolitical noise," I have narrowed the scope of this analysis to The Guardian and The Kyiv Independent.

In [22]:
news = df.query("source in ['theguardian.com', 'kyivindependent.com']").copy()

In [23]:
df.shape

(3856, 4)

In [27]:
news.shape

(2156, 4)

In [28]:
news['headers'] = [fetch_article_headline(url) for url in tqdm(news['url'], desc="Processing URLs")]

Processing URLs:   0%|          | 0/2156 [00:00<?, ?it/s]

In [29]:
news.head()

,event_date,source,url,themes,headers
0,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_FNCACT_CHANCELLOR,17;TAX_FNCACT_CHANCELLOR...",Ukraine war briefing: Merz tells Trump Ukraine...
1,2026-03-04,theguardian.com,https://www.theguardian.com/world/2026/mar/04/...,"TAX_ETHNICITY_AMERICANS,5192;WB_508_POWER_SYST...",Middle East conflict offers economic lifeline ...
2,2026-03-03,theguardian.com,https://www.theguardian.com/stage/2026/mar/03/...,"CLOSURE,2139;TAX_ETHNICITY_KOREAN,1693;TAX_WOR...",Ukraine Unbroken review – five searing dramas ...
3,2026-03-03,kyivindependent.com,https://kyivindependent.com/ukraine-backs-us-i...,"TAX_ETHNICITY_IRANIANS,2755;TAX_ETHNICITY_IRAN...",How will Trump's war against Iran impact Ukraine?
4,2026-03-03,theguardian.com,https://www.theguardian.com/world/2026/mar/03/...,"TAX_FNCACT_AUTHORITIES,2679;EPU_POLICY_AUTHORI...",Ukraine war briefing: Russia’s army records sl...


In [30]:
news.event_date = pd.to_datetime(news.event_date)

In [43]:
news.groupby(['event_date', 'source'], as_index = False)\
                    .agg({'headers': list}).head()

,event_date,source,headers
0,2025-01-01,theguardian.com,[Ukraine war briefing: Zelenskyy vows his coun...
1,2025-01-02,kyivindependent.com,[Slovakia may cut aid to Ukrainian refugees ov...
2,2025-01-02,theguardian.com,[Russian gas shutdown forces closure of almost...
3,2025-01-03,kyivindependent.com,['Trump can be decisive in ending war' — Zelen...
4,2025-01-03,theguardian.com,[Ukraine war briefing: Zelenskyy hopes Trump’s...


In [41]:
news[news.headers.str.contains('Error')].head()

,event_date,source,url,themes,headers
296,2026-01-16,kyivindependent.com,https://kyivindependent.com/ukraine-war-latest...,"TAX_FNCACT_CREW_MEMBER,3049;TAX_FNCACT_WOMAN,3...",Request Error: 404 Client Error: Not Found for...
787,2025-11-21,kyivindependent.com,https://kyivindependent.com/kraine-war-latest-...,"CRISISLEX_O02_RESPONSEAGENCIESATCRISIS,21;TAX_...",Request Error: 404 Client Error: Not Found for...
2793,2025-03-11,kyivindependent.com,https://kyivindependent.com/moscow-targeted-in...,"TAX_FNCACT_EDITOR,6;DRONES,104;DRONES,292;TAX_...",Request Error: 404 Client Error: Not Found for...


I've decided to remove those 3 error headers to keep my data clean

In [44]:
news = news[~news.headers.str.contains('Error')]

In [47]:
news_final = news.groupby(['event_date', 'source'], as_index = False)\
                    .agg({'headers': list})

In [48]:
news_final.head()

,event_date,source,headers
0,2025-01-01,theguardian.com,[Ukraine war briefing: Zelenskyy vows his coun...
1,2025-01-02,kyivindependent.com,[Slovakia may cut aid to Ukrainian refugees ov...
2,2025-01-02,theguardian.com,[Russian gas shutdown forces closure of almost...
3,2025-01-03,kyivindependent.com,['Trump can be decisive in ending war' — Zelen...
4,2025-01-03,theguardian.com,[Ukraine war briefing: Zelenskyy hopes Trump’s...


Final Filtering and Database Integration

Key actions taken:

Relevance Filtering: I have curated a daily feed of headlines from trusted sources (The Guardian and The Kyiv Independent). This allows for a much more granular analysis of how specific world events and front-line news correlate with aid dynamics.

In [55]:
import sqlite3
import pandas as pd
from pathlib import Path

# 1. Setup path
db_path = Path.cwd().parent / "data" / "master" / "master.db"
db_path.parent.mkdir(parents=True, exist_ok=True)

# 2. Refine columns and types
# Renaming to 'date' and stripping time components in one pass
news_export = news_final.rename(columns={'event_date': 'date'})
news_export['date'] = pd.to_datetime(news_export['date']).dt.date.astype(str)
news_export['headers'] = news_export['headers'].astype(str)

# 3. Export using 'with' to auto-close connection
with sqlite3.connect(db_path) as conn:
    news_export.to_sql('news', conn, if_exists='replace', index=False)
    print(f"--- EXPORT SUCCESSFUL: {len(news_export)} rows saved to 'news' table ---")

--- EXPORT SUCCESSFUL: 619 rows saved to 'news' table ---


I realized that storing every single news article as a separate row in the database was not the most efficient approach for daily correlation analysis. To improve this, I restructured the data:

Daily Aggregation: Instead of individual entries, I grouped the news by event_date and source. Now, each row in the database represents a single day for a specific source, with all headlines stored as a list.

Noise Reduction: I filtered out the few remaining scraping errors before grouping to ensure that our headline lists contain only valid, human-readable information.

Schema Update: The master.db now contains a more compact and meaningful news table, optimized for join operations with the daily donation datasets.